In [1]:
!pip install -q transformers==4.51.0 tokenizers==0.21.2

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score, classification_report

# =========================
# 1. PATH
# =========================
TRAIN_PATH = "TRAIN_PATH"
VAL_PATH   = "VAL_PATH"
TEST_PATH  = "TEST_PATH"

MODEL_PATH = "nd-khoa/vihsd-uit-visobert-v2"
TOKENIZER_PATH = "nd-khoa/vihsd-uit-visobert-v2"

# =========================
# 2. LOAD DATA
# =========================
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

def convert_binary(label):
    if label == 0:
        return 0       # Clean
    else:
        return 1       # Offensive + Hate = Toxic


train_df["binary_label"] = train_df["labels"].apply(convert_binary)
val_df["binary_label"]   = val_df["labels"].apply(convert_binary)
test_df["binary_label"]  = test_df["labels"].apply(convert_binary)


print("\nTRAIN:")
print(train_df["binary_label"].value_counts())

print("\nVAL:")
print(val_df["binary_label"].value_counts())

print("\nTEST:")
print(test_df["binary_label"].value_counts())

# đảm bảo binary
for df in [train_df, val_df, test_df]:
    df["binary_label"] = df["binary_label"].astype(int)

print(train_df["binary_label"].value_counts())
print(val_df["binary_label"].value_counts())
print(test_df["binary_label"].value_counts())

# =========================
# 3. TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_PATH
)

class ToxicDataset(Dataset):
    def __init__(self, df):
        self.texts = df["texts"].fillna("").astype(str).tolist()
        self.labels = df["binary_label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=256
        )

        enc["labels"] = self.labels[idx]

        return {
            k: torch.tensor(v)
            for k, v in enc.items()
        }

train_dataset = ToxicDataset(train_df)
val_dataset   = ToxicDataset(val_df)
test_dataset  = ToxicDataset(test_df)

# =========================
# 4. MODEL
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=2,
    ignore_mismatched_sizes=True
)

print("Parameters:",
      sum(p.numel() for p in model.parameters()))

# =========================
# 5. METRICS
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "toxic_f1": f1_score(labels, preds, pos_label=1)
    }

# =========================
# 6. TRAINING
# =========================
training_args = TrainingArguments(
    output_dir="./visobert_binary",

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_steps=100,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

# =========================
# 7. TRAIN
# =========================
trainer.train()

# =========================
# 8. TEST
# =========================
pred = trainer.predict(test_dataset)

y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)

print("\n===== TEST =====")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Macro-F1 :", f1_score(y_true, y_pred, average="macro"))
print("Toxic F1 :", f1_score(y_true, y_pred, pos_label=1))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Clean", "Toxic"]
    )
)

# =========================
# 9. SAVE
# =========================
trainer.save_model("./visobert_binary_best_audit")
tokenizer.save_pretrained("./visobert_binary_best_audit")


TRAIN:
binary_label
0    7985
1    6040
Name: count, dtype: int64

VAL:
binary_label
0    2121
1     957
Name: count, dtype: int64

TEST:
binary_label
0    5459
1    1898
Name: count, dtype: int64
binary_label
0    7985
1    6040
Name: count, dtype: int64
binary_label
0    2121
1     957
Name: count, dtype: int64
binary_label
0    5459
1    1898
Name: count, dtype: int64


config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at nd-khoa/vihsd-uit-visobert-v2 and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parameters: 97567490


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Toxic F1
1,0.266500,0.159826,0.936647,0.927429,0.901565
2,0.103200,0.246837,0.937947,0.928418,0.902302
3,0.019000,0.298555,0.950292,0.942908,0.922374



===== TEST =====
Accuracy : 0.9025417969280957
Macro-F1 : 0.8788029697605544
Toxic F1 : 0.825164594001463

Classification Report:
              precision    recall  f1-score   support

       Clean       0.96      0.91      0.93      5459
       Toxic       0.77      0.89      0.83      1898

    accuracy                           0.90      7357
   macro avg       0.86      0.90      0.88      7357
weighted avg       0.91      0.90      0.90      7357



('./visobert_binary_best_audit/tokenizer_config.json',
 './visobert_binary_best_audit/special_tokens_map.json',
 './visobert_binary_best_audit/sentencepiece.bpe.model',
 './visobert_binary_best_audit/added_tokens.json',
 './visobert_binary_best_audit/tokenizer.json')